In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.9 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# LiNiO₂ on ECHIDNA

&nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; 

Fit the layered oxide LiNiO₂ to constant-wavelength neutron powder data from ECHIDNA at ANSTO.
Li and Ni mix between two sites, held to full occupancy by constraints, and the profile has
Finger-Cox-Jephcoat asymmetry.

The project is the catalogue case `pd-neut-cwl_lmo-echidna`, the one the command line and the
app's Examples open. Li and Ni share their sites through five relations (displacements and
occupancies), kept from the start.

In [2]:
import easydiffraction as edi

## Load the project

In [3]:
project_path = edi.download_data('pd-neut-cwl_lmo-echidna', destination='data')
project = edi.Project.load(project_path)
structure = project.structures['lmo']
experiment = project.experiments['echidna']
project.display.constraints()

Data 'pd-neut-cwl_lmo-echidna' is at data/pd-neut-cwl_lmo-echidna-134fba941254/pd-neut-cwl_lmo-echidna


View(html=<79,456 characters>, height=460)

The project marks what its fit frees. The fit below frees it in stages, so start with all of it
fixed.

In [4]:
for parameter in project.free_parameters:
    parameter.free = False

In [5]:
project.display.pattern('echidna')

View(html=<451,720 characters>, height=480)

## Perform the analysis

The fit goes in stages, freeing one group of parameters at a time.

### Scale, 2θ zero and cell

The scale, the zero shift and the two cell lengths put the peaks in place and at the right
height.

In [6]:
experiment.linked_structures['lmo'].scale.free = True
experiment.instrument.calib_twotheta_offset.free = True
structure.cell.length_a.free = True
structure.cell.length_c.free = True

In [7]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,lmo,cell,,length_a,2.88,,0,30,Å
2,lmo,cell,,length_c,14.18,,0,30,Å
3,echidna,instrument,,calib_twotheta_offset,0,,-inf,inf,deg
4,echidna,linked_structure,lmo,scale,10,,-inf,inf,


In [8]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_lmo-echidna, 3002 pts fitted / 3200 loaded, 4 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       29.38        81.99
      1      0.020      8.05         6.16   92.5% ↓
      2      0.025      8.05         6.16   
  done · converged · 2 iters · reduced χ² 6.16 · Rwp 8.05 % · 0.03 s


View(html=<84,771 characters>, height=376)

In [9]:
project.display.pattern('echidna')

View(html=<453,390 characters>, height=480)

### Sample displacement and transparency

The sample sits a little off the diffractometer axis and the beam goes into it; both shift the
peaks with angle, unlike the zero.

In [10]:
experiment.instrument.calib_sample_displacement.free = True
experiment.instrument.calib_sample_transparency.free = True

In [11]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_lmo-echidna, 3002 pts fitted / 3200 loaded, 6 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        8.05         6.16
      1      0.022      8.04         6.14   0.3% ↓
  done · converged · 1 iters · reduced χ² 6.14 · Rwp 8.04 % · 0.03 s


View(html=<86,848 characters>, height=376)

In [12]:
project.display.pattern('echidna')

View(html=<453,492 characters>, height=480)

### Background

The nine background points follow the level between the peaks.

In [13]:
for point in experiment.background:
    point.intensity.free = True

In [14]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_lmo-echidna, 3002 pts fitted / 3200 loaded, 15 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        8.04         6.16
      1      0.029      6.46         3.98   35.4% ↓
  done · converged · 1 iters · reduced χ² 3.98 · Rwp 6.46 % · 0.03 s


View(html=<99,325 characters>, height=376)

In [15]:
project.display.pattern('echidna')

View(html=<482,284 characters>, height=480)

### Peak widths

In [16]:
experiment.peak.broad_gauss_u.free = True
experiment.peak.broad_gauss_v.free = True
experiment.peak.broad_gauss_w.free = True
experiment.peak.broad_lorentz_y.free = True

In [17]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_lmo-echidna, 3002 pts fitted / 3200 loaded, 19 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        6.46         3.99
      1      0.028      6.39         3.90   2.2% ↓
  done · converged · 1 iters · reduced χ² 3.90 · Rwp 6.39 % · 0.03 s


View(html=<105,108 characters>, height=376)

In [18]:
project.display.pattern('echidna')

View(html=<482,189 characters>, height=480)

### Oxygen position

O's height along c is the only coordinate the symmetry leaves free.

In [19]:
structure.atom_sites['O'].fract_z.free = True

In [20]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_lmo-echidna, 3002 pts fitted / 3200 loaded, 20 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        6.39         3.90
      1      0.034      6.20         3.67   6.0% ↓
  done · converged · 1 iters · reduced χ² 3.67 · Rwp 6.20 % · 0.04 s


View(html=<106,483 characters>, height=376)

In [21]:
project.display.pattern('echidna')

View(html=<482,161 characters>, height=480)

### Displacements

Li1's, Ni2's and O's; Ni1 and Li2 follow by the relations.

In [22]:
structure.atom_sites['Li1'].adp_iso.free = True
structure.atom_sites['Ni2'].adp_iso.free = True
structure.atom_sites['O'].adp_iso.free = True

In [23]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_lmo-echidna, 3002 pts fitted / 3200 loaded, 23 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        6.20         3.67
      1      0.031      5.31         2.70   26.6% ↓
      2      0.040      5.31         2.70   
  done · converged · 2 iters · reduced χ² 2.70 · Rwp 5.31 % · 0.05 s


View(html=<110,541 characters>, height=376)

In [24]:
project.display.pattern('echidna')

View(html=<482,076 characters>, height=480)

### Li/Ni mixing

Last, Li1's occupancy, which sets how much Ni sits on the Li site; it moves with the
displacements, so it comes after them.

In [25]:
structure.atom_sites['Li1'].occupancy.free = True

In [26]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_lmo-echidna, 3002 pts fitted / 3200 loaded, 24 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        5.31         2.70
      1      0.040      4.99         2.39   11.5% ↓
  done · converged · 1 iters · reduced χ² 2.39 · Rwp 4.99 % · 0.05 s


View(html=<111,605 characters>, height=376)

In [27]:
project.display.pattern('echidna')

View(html=<482,054 characters>, height=480)

In [28]:
project.display.fit.correlations()

,1,2,3,4
1 lmo.cell.length_a,,,,
2 echidna.instrument.calib_twotheta_offset,0.89,,,
3 echidna.instrument.calib_sample_displacement,-0.94,-0.90,,
4 echidna.instrument.calib_sample_transparency,-0.74,-0.95,0.75,


In [29]:
project.display.structure('lmo')

View(html=<4,415,420 characters>, height=660)

## Save the project

In [30]:
project.save_as('pd-neut-cwl_lmo-echidna')
project.report.save_html('pd-neut-cwl_lmo-echidna.html')

Saving project to 'pd-neut-cwl_lmo-echidna'
Saving report to 'pd-neut-cwl_lmo-echidna.html'
